# 01 — Exploring the Raw Data

Before any cleaning, we profile the two raw datasets in `data/raw/` to understand their shape, quality, and quirks. All operations here are read-only with respect to the source files.

- **Primary** — `spotify_tracks.csv`: audio features and a proprietary popularity score, with a per-track genre label.
- **Secondary** — `tracks_1921_2020.csv`: the same audio features plus a release date (no genre), used for decade-level trends.

## Primary dataset — Spotify Tracks

*Source: maharshipandya/-spotify-tracks-dataset (Kaggle).* Used to study what drives a track's popularity and how audio profiles differ by genre.

### Load and inspect

The first column is an unnamed running index, so we read it in as the DataFrame index rather than a data column.

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/spotify_tracks.csv", index_col=0)

print(df.shape)
df.head()

(114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


### Duplicate tracks across genres

The dataset is assembled by sampling per-genre playlists, so a track that sits in multiple genres is repeated once per genre. "One row = one track" does not hold here, which has direct consequences for any statistic computed on the raw file.

In [3]:
n_rows = len(df)
n_tracks = df["track_id"].nunique()

print(f"rows:            {n_rows:,}")
print(f"unique tracks:   {n_tracks:,}")
print(f"duplicate rows:  {n_rows - n_tracks:,}")

rows:            114,000
unique tracks:   89,741
duplicate rows:  24,259


In [4]:
# The most-repeated track: identical audio features and popularity, differing only in track_genre.

most_dup = df["track_id"].value_counts().index[0]
df.loc[df["track_id"] == most_dup,
       ["track_name", "artists", "popularity", "energy", "valence", "track_genre"]]

,track_name,artists,popularity,energy,valence,track_genre
8315,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,blues
19759,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,country
34728,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,folk
62226,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,j-pop
63087,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,j-rock
82064,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,power-pop
84129,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,psych-rock
99727,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,singer-songwriter
102732,Baby Blue - Remastered 2010,Badfinger,67,0.876,0.515,songwriter


### Data quality: missing and impossible values

Two checks before trusting the data: how many values are missing, and how many are physically impossible (extraction artifacts rather than genuine outliers).

In [5]:
# Count missing values per column; show only columns that actually have any.
missing = df.isna().sum()
missing[missing > 0]

artists       1
album_name    1
track_name    1
dtype: int64

In [6]:
# Inspect the actual rows that have any missing value.
df[df.isna().any(axis=1)]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [7]:
# Physically impossible values: tempo of 0, sub-30s or >30min durations, invalid time signature.
checks = {
    "tempo == 0 (beat detection failed)":     (df["tempo"] == 0).sum(),
    "duration < 30s (skits / stingers)":      (df["duration_ms"] < 30_000).sum(),
    "duration > 30min (audiobooks / mixes)":  (df["duration_ms"] > 1_800_000).sum(),
    "time_signature == 0 (invalid)":          (df["time_signature"] == 0).sum(),
}
for label, count in checks.items():
    print(f"{label:40s} {count:>4} rows")

tempo == 0 (beat detection failed)        157 rows
duration < 30s (skits / stingers)          17 rows
duration > 30min (audiobooks / mixes)      36 rows
time_signature == 0 (invalid)             163 rows


In [8]:
# Inspect the actual rows with impossible values.

impossible = (
    (df["tempo"] == 0)
    | (df["duration_ms"] < 30_000)
    | (df["duration_ms"] > 1_800_000)
    | (df["time_signature"] == 0)
)

df.loc[impossible, ["track_name", "artists", "duration_ms", "tempo", "time_signature", "track_genre"]].head(10)


,track_name,artists,duration_ms,tempo,time_signature,track_genre
2926,Sanki Yapamadım,Yaşlı Amca,213198,138.616,0,alt-rock
4131,The Departure,Max Richter;Lang Lang,151506,0.000,0,ambient
4379,The End of Childhood (feat. Jack Liebeck),Dario Marianelli;Jack Liebeck;Benjamin Wallfisch,73266,0.000,0,ambient
4664,Ferme Les Yeux,Sylvain Chauveau,68493,0.000,0,ambient
10191,Welcome To The Jungle - Continuous DJ Mix Pt. 1,Deekline;Ed Solo,2733257,87.499,4,breakbeat
10739,"Welcome To The Jungle - Continuous DJ Mix, Pt. 1",Deekline;Ed Solo;Serial Killaz,3274999,175.009,4,breakbeat
10796,Welcome To The Jungle Vol. 2 - Continuous DJ M...,Ed Solo;Deekline,2523428,175.005,4,breakbeat
10935,Crossing Wires 002 - Continuous DJ Mix,Timo Maas,4789026,121.055,4,breakbeat
10949,"Bass Shakers 2015 - Continuous DJ Mix, Pt. 1",Lady Waks,2796984,130.572,4,breakbeat
10984,Crossing Wires 002 - Continuous DJ Mix,Timo Maas,4789026,121.055,4,breakbeat


## Secondary dataset — Historical tracks (1921–2020)

*Source: yamaerenay/spotify-dataset-19212020-600k-tracks (Kaggle).* Shares the same audio features but adds a **release date** and has **no genre**; it powers the decade-level trend analysis. The two datasets are not joined — their popularity scores are proprietary snapshots taken years apart and are not comparable.

### Load and inspect

No `index_col` here: the first column (`id`) is a meaningful track identifier, not a running index.

In [9]:
hist = pd.read_csv("../data/raw/tracks_1921_2020.csv")

print(hist.shape)
hist.head(3)

(586672, 20)


,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.445,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.263,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.177,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5


### Release dates and decades

`release_date` comes in mixed formats, so we take the first four characters as the year and floor to the decade. Note the uneven sample sizes across decades — trend charts must use averages, not counts, and flag the sparse early decades.

In [10]:
# release_date has mixed formats. String length tells them apart:
# 4 = year only ("1998"), 7 = year-month, 10 = full date.
hist["release_date"].str.len().value_counts().sort_index()

release_date
4     136489
7       2102
10    448081
Name: count, dtype: int64

In [11]:
# Take the first 4 characters as the year, derive the decade, and count tracks per decade.
year = hist["release_date"].str[:4].astype(int)
decade = (year // 10) * 10

print(f"year range: {year.min()}–{year.max()}\n")
decade.value_counts().sort_index()

year range: 1900–2021



release_date
1900         1
1920      7610
1930     13037
1940     18042
1950     35370
1960     47270
1970     61841
1980     82322
1990    108875
2000     86841
2010    105245
2020     20218
Name: count, dtype: int64